# 08 -- Final Summary and Paper Figures

Consolidated run summary and full decision history, then the remaining paper
figures: representative per-pattern graph panels, the Layer 1 PCA-scatter proof
figure, and the polished Section 4.5 routing-ablation figure.

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import Patch

from awik.config import ANSWER_SET, MALICIOUS_USERS, DEFAULT_K, DEFAULT_MAH_THRESH
from awik.neo4j_client import Neo4jConnection
from awik import io

df_master = io.load_df('01_df_master')
df_semua = io.load_df('03_df_semua')
df_threats = io.load_df('03_df_threats')
df_ranking = io.load_df('03_df_ranking')
state03 = io.load_json('03_layer1_state')
N_MASTER = state03['N_MASTER']

df_surges = io.load_df('04_df_surges')
df_forensic = io.load_df('04_df_forensic')
state04 = io.load_json('04_layer2_state')
users_targeted = set(state04['users_targeted'])

df_final = io.load_df('05_df_final')
state05 = io.load_json('05_evaluation_state')
h_l1, h_gc = state05['h_l1'], state05['h_gc']

df_sweep = io.load_df('07_df_sweep')
df_seed = io.load_df('07_df_seed')
c1 = pd.read_csv('../data/interim/06_c1_awik.csv')['weeks']
c2 = pd.read_csv('../data/interim/06_c2_kmeans.csv')['weeks']
try:
    df_paradox = io.load_df('07_df_paradox')
except FileNotFoundError:
    df_paradox = pd.DataFrame()

## Final summary + full decision history

In [ ]:
df_master.to_csv('../results/tables/AWIK_MasterFeatures.csv', index=False)
df_threats.to_csv('../results/tables/AWIK_Layer1.csv', index=False)
df_ranking.to_csv('../results/tables/AWIK_Ranking.csv', index=False)
if not df_surges.empty:
    df_surges.to_csv('../results/tables/AWIK_Baseline.csv', index=False)
if not df_forensic.empty:
    df_forensic.to_csv('../results/tables/AWIK_Forensic.csv', index=False)
df_final.to_csv('../results/tables/AWIK_FinalSuspects.csv', index=False)

pop_final = df_master['User_ID'].nunique()
prevalensi = len(MALICIOUS_USERS) / pop_final * 100

print("=" * 70)
print("AWIK -- FINAL SUMMARY")
print("=" * 70)
print(f"\nDataset        : CERT Insider Threat r6.2")
print(f"Users          : {pop_final:,}")
print(f"Sliding Windows: 76 weeks")
print(f"Answer Set     : {len(MALICIOUS_USERS)} actors of {pop_final:,} accounts "
      f"({prevalensi:.2f}% population) -- extreme low prevalence. Mandatory context")
print(f"                 for reading any metric below.")
print(f"\nMahalanobis threshold: {DEFAULT_MAH_THRESH} (empirical, fixed, label-free; "
      f"NOT theoretical chi2.ppf() -- see docs/methodology.md)")
print(f"\nLayer 1 Recall : {h_l1['Recall']:.4f} ({h_l1['TP']}/6 actors)")
print(f"Graph-conf Rec : {h_gc['Recall']:.4f} ({h_gc['TP']}/6 actors)")
print(f"Routing pool   : {len(users_targeted)} users ({len(users_targeted)/pop_final*100:.1f}% population)")
print(f"Flagging Rate  : {len(df_threats)/N_MASTER*100:.2f}%")

if not df_paradox.empty:
    print(f"\nCovariance-Aligned Evasion:")
    print(f"  mean axis_ratio          : {df_paradox['axis_ratio'].mean():.2f}")
    print(f"  alignment > 0.7          : {(df_paradox['align_major']>0.7).mean()*100:.1f}% of weeks")
    print(f"  mean Mah^2               : {df_paradox['Mah_sq'].mean():.3f} (threshold {DEFAULT_MAH_THRESH})")

print(f"\nChronic vs. Scattered:")
print(f"  AWIK   : {c1.mean():.2f} weeks/user, {(c1>=5).mean()*100:.1f}% chronic")
print(f"  KMeans : {c2.mean():.2f} weeks/user, {(c2>=5).mean()*100:.1f}% chronic")

print("\n" + "=" * 70)
print("DECISION HISTORY -- ARCHITECTURE & THRESHOLD (full detail: docs/methodology.md)")
print("=" * 70)
print("Layer 1 threshold:")
print("  1) theoretical chi-square (9.21)   -> ABANDONED, normality assumption disproven")
print("  2) Yeo-Johnson power transform      -> REJECTED, made kurtosis worse")
print("  3) manual percentiles (95/97/99)    -> REJECTED, label-aware")
print("  4) knee-detection                   -> REJECTED, 14.6% of points missed")
print(f"  5) empirical p99 ({DEFAULT_MAH_THRESH})              -> USED, label-free")
print("\nLayer 1 architecture:")
print("  A) K-Means->DBSCAN (original design) -> KEPT")
print("  B) DBSCAN->K-Means                   -> equivalent to A, no improvement")
print("  C) Voting ensemble                   -> REJECTED, worst persistence")
print("  D) K-Means->K-Means                  -> equivalent to A")
print("  E) DBSCAN->DBSCAN (arbitrary)        -> initially ahead, but parameters not label-free")
print("  E-v2) DBSCAN->DBSCAN (label-free)    -> REJECTED, 21.4% flagging rate, over-flagging")
print("  EKS-4/5) Isolation Forest & LOF       -> REJECTED as sole Layer 1, huge FP without routing")
print("\nConclusion: design A is the only configuration that stays reasonable once every")
print("relevant parameter is derived label-free, consistently.")

## Figure: representative suspect per pattern (G1, G2, G3, G6)

In [ ]:
import os
db_conn = Neo4jConnection("bolt://localhost:7687", "neo4j", os.environ.get("NEO4J_PASSWORD", "changeme"))

FIGURE_SUSPECTS = {
    'G1': 'AAF0819',   # representative, not a verified Answer Set actor
    'G2': 'ACM2278',   # confirmed malicious insider, Scenario 1
    'G3': 'DAH3324',   # representative, not a verified Answer Set actor
    'G6': 'CDE1846',   # confirmed malicious insider, Scenario 4
}

def fetch_g1(uid):
    q = """
    MATCH (suspect:User {user_id: $uid})-[r1:LOGGED_ON_TO]->(pc:PC)
          <-[r2:LOGGED_ON_TO]-(owner:User)
    WHERE owner.user_id <> $uid
      AND (r1.timestamp.hour >= 18 OR r1.timestamp.hour <= 6)
    RETURN suspect.user_id AS Suspect, pc.pc_id AS PC, owner.user_id AS Owner
    ORDER BY r1.timestamp LIMIT 200
    """
    return db_conn.query_to_dataframe(q, {"uid": uid})

def fetch_g2(uid):
    q = """
    MATCH (suspect:User {user_id: $uid})-[r1:LOGGED_ON_TO]->(pc:PC)
          <-[r2:LOGGED_ON_TO]-(owner:User)
    WHERE owner.user_id <> $uid
    MATCH (suspect)-[r3:SENT_EMAIL]->(email_node)
    WHERE r3.timestamp > r1.timestamp
      AND r3.timestamp < r1.timestamp + duration({hours: 24})
      AND r3.size > 50000 AND r3.activity = 'Send'
    RETURN suspect.user_id AS Suspect, pc.pc_id AS PC, owner.user_id AS Owner, email_node.address AS Email
    ORDER BY r1.timestamp LIMIT 200
    """
    return db_conn.query_to_dataframe(q, {"uid": uid})

def fetch_g3(uid):
    q = """
    MATCH (suspect:User {user_id: $uid})-[r1:LOGGED_ON_TO]->(pc:PC)
          <-[r2:LOGGED_ON_TO]-(owner:User)
    WHERE owner.user_id <> $uid
      AND r1.timestamp.hour >= 7 AND r1.timestamp.hour <= 17
    MATCH (suspect)-[r3:SENT_EMAIL]->(email_node)
    WHERE r3.size > 100000 AND r3.activity = 'Send'
    RETURN suspect.user_id AS Suspect, pc.pc_id AS PC, owner.user_id AS Owner, email_node.address AS Email
    LIMIT 200
    """
    return db_conn.query_to_dataframe(q, {"uid": uid})

def fetch_g6(uid):
    q = """
    MATCH (suspect:User {user_id: $uid})-[r1:LOGGED_ON_TO]->(pc:PC)
    WHERE r1.timestamp.hour >= 18 OR r1.timestamp.hour <= 6
    WITH suspect, collect(DISTINCT pc) AS candidate_pcs
    UNWIND candidate_pcs AS pc
    MATCH (any_user:User)-[r2:LOGGED_ON_TO]->(pc)
    WITH suspect, pc, any_user, count(r2) AS n_login
    ORDER BY pc, n_login DESC
    WITH suspect, pc, collect({u: any_user, n: n_login})[0] AS top, sum(n_login) AS total
    WITH suspect, pc, top.u AS owner, toFloat(top.n) / total AS share
    WHERE share >= 0.5 AND owner <> suspect
    RETURN suspect.user_id AS Suspect, pc.pc_id AS PC, owner.user_id AS Owner
    LIMIT 200
    """
    return db_conn.query_to_dataframe(q, {"uid": uid})

fetchers = {'G1': fetch_g1, 'G2': fetch_g2, 'G3': fetch_g3, 'G6': fetch_g6}
titles = {'G1': 'G1: After-Hours Lateral Login', 'G2': 'G2: Lateral -> Mass Email',
          'G3': 'G3: Daytime Lateral + Large Email', 'G6': 'G6: Attacker-Centric Lateral Movement'}

def draw_panel(df, ax, title):
    if df is None or df.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', color='gray')
        ax.axis('off'); ax.set_title(title, fontsize=10, fontweight='bold')
        return
    G = nx.DiGraph()
    suspects = set(df['Suspect'].unique())
    for _, row in df.iterrows():
        G.add_edge(row['Suspect'], row['PC'])
        if 'Owner' in row and pd.notna(row['Owner']):
            G.add_edge(row['Owner'], row['PC'])
        if 'Email' in row and pd.notna(row['Email']):
            G.add_edge(row['Suspect'], row['Email'])
    pos = nx.spring_layout(G, k=1.2, seed=42)
    node_colors = ['#2ca02c' if n in suspects else '#a6d8a6' for n in G.nodes()]
    node_sizes = [900 if n in suspects else 500 for n in G.nodes()]
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=node_sizes, edgecolors='white', linewidths=1.2)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='gray', alpha=0.6, arrowsize=8)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=6)
    ax.set_title(title, fontsize=10, fontweight='bold'); ax.axis('off')

fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
for ax, (g_label, uid) in zip(axes, FIGURE_SUSPECTS.items()):
    df = fetchers[g_label](uid)
    print(f"{g_label} ({uid}): {len(df)} rows, {df['PC'].nunique() if not df.empty and 'PC' in df else 0} unique PCs")
    draw_panel(df, ax, titles[g_label])

fig.legend(handles=[Patch(facecolor='#2ca02c', label='Suspect'),
                     Patch(facecolor='#a6d8a6', label='Owner / Target / PC / Email')],
           loc='lower center', ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.savefig('../results/figures/AWIK_Figure_GraphPatterns.png', dpi=200, bbox_inches='tight')
plt.show()
db_conn.close()

## Figure: Layer 1 process, PCA scatter proof

Connects Section 4 back to the Section 3 architecture figure -- shows K-Means ->
DBSCAN -> Mahalanobis (empirical threshold) actually running on real data. Sample
week 2011-W10 -- the overlap range of CMP2946 (active 2011 W5-13) and CDE1846
(active 2011 W8-17), both actors' real attack weeks.

In [ ]:
yr_demo, wk_demo = 2011, 10
df_demo = df_semua[(df_semua['Year']==yr_demo) & (df_semua['Week']==wk_demo)].copy()

fig, ax = plt.subplots(figsize=(7, 6))
cluster_colors = plt.cm.tab10(np.linspace(0, 1, DEFAULT_K))

for cid in range(DEFAULT_K):
    mask = (df_demo['Macro_Role'] == cid) & (df_demo['AWIK_Flag'] == 0)
    ax.scatter(df_demo.loc[mask, 'PC1'], df_demo.loc[mask, 'PC2'], s=22,
               color=cluster_colors[cid], alpha=0.35, label=f'Cluster {cid} (normal)')

flagged = df_demo[df_demo['AWIK_Flag'] == 1]
ax.scatter(flagged['PC1'], flagged['PC2'], s=70, facecolors='none', edgecolors='#d62728',
           linewidths=1.4, alpha=0.7, zorder=4, label='AWIK_Flag=1 (Layer 1 anomaly)')

for actor in sorted(MALICIOUS_USERS):
    if (yr_demo, wk_demo) not in ANSWER_SET[actor]['active_weeks']:
        continue
    row = df_demo[df_demo['User_ID'] == actor]
    if row.empty:
        print(f"[WARNING] {actor}: active week but no row in df_semua "
              f"({yr_demo}-W{wk_demo:02d}) -- check the len(df_c)<10 filter in run_awik_pipeline.")
        continue
    ax.scatter(row['PC1'], row['PC2'], s=260, marker='*', color='gold', edgecolors='black',
               linewidths=1.3, zorder=5, label=f'{actor} (real attack week)')
    print(f"{actor}: AWIK_Flag={int(row['AWIK_Flag'].iloc[0])} "
          f"| Mahalanobis_Sq={float(row['Mahalanobis_Sq'].iloc[0]):.3f} (threshold={DEFAULT_MAH_THRESH}) "
          f"| DBSCAN_Label={int(row['DBSCAN_Label'].iloc[0])}")

ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title(f'Layer 1 Process (Two-Stage Spatial Clustering) -- {yr_demo}-W{wk_demo:02d}\n'
             f'K-Means (k={DEFAULT_K}) partitions macro-role; DBSCAN + empirical Mahalanobis flags anomalies')
ax.legend(fontsize=7.5, loc='best', framealpha=0.9)
plt.tight_layout()
plt.savefig('../results/figures/AWIK_Layer1_PCA_Demo.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nPoints this week: {len(df_demo)} | AWIK_Flag=1: {len(flagged)} ({len(flagged)/len(df_demo)*100:.1f}%)")

## Figure: adaptive-baseline routing ablation (Section 4.5, polished)

In [ ]:
h_full_asli, h_full_ablasi = state05['h_full_asli'], state05['h_full_ablasi']
g1_suspects = set(state04['g1_suspects']); g2_suspects = set(state04['g2_suspects'])
g3_suspects = set(state04['g3_suspects']); g5_suspects = set(state04['g5_suspects'])
g6_suspects = set(state04['g6_suspects'])
g1_a=len(set(state04['g1_suspects_a'])); g2_a=len(set(state04['g2_suspects_a']))
g3_a=len(set(state04['g3_suspects_a'])); g5_a=len(set(state04['g5_suspects_a'])); g6_a=len(set(state04['g6_suspects_a']))

conditions = ['Routing\nactive', 'Routing\ndisabled']
recall_vals = [h_full_asli['Recall'], h_full_ablasi['Recall']]
bar_colors = ['#2196F3', '#EF5350']
queries = ['G1', 'G2', 'G3', 'G4', 'G5', 'G6']
vol_active = [len(g1_suspects), len(g2_suspects), len(g3_suspects), 0, len(g5_suspects), len(g6_suspects)]
vol_ablasi = [g1_a, g2_a, g3_a, 0, g5_a, g6_a]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), gridspec_kw={'width_ratios': [1, 1.6]})
fig.suptitle('Ablation Study: Adaptive-Baseline Routing\nCERT r6.2 -- Full System Recall and Query Volume',
             fontsize=11, fontweight='bold', y=1.02)

ax = axes[0]
bars = ax.bar(conditions, recall_vals, width=0.45, color=bar_colors, edgecolor='white', linewidth=0.8)
ax.set_ylabel('Full System Recall', fontsize=10); ax.set_ylim(0, 1.1)
ax.set_title('Full-System Recall', fontsize=10); ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
for bar, val in zip(bars, recall_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax2 = axes[1]
x = np.arange(len(queries)); w = 0.35
ax2.bar(x - w/2, vol_active, w, color='#2196F3', label='Routing active', edgecolor='white')
ax2.bar(x + w/2, vol_ablasi, w, color='#EF5350', label='Routing disabled', edgecolor='white')
ax2.set_yscale('log'); ax2.set_ylabel('Accounts examined (log scale)', fontsize=9)
ax2.set_title('Accounts per query -- active vs disabled\n(G4 permanently disabled; G6 admits a clean single-variable ablation)', fontsize=9)
ax2.set_xticks(x); ax2.set_xticklabels(queries, fontsize=10)
ax2.grid(axis='y', alpha=0.3); ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
ax2.axvspan(5 - 0.5, 5 + 0.5, alpha=0.08, color='gold', zorder=0)
ax2.legend(fontsize=9, loc='upper left')

plt.tight_layout()
plt.savefig('../results/figures/AWIK_Ablation_Routing_Chart.pdf', bbox_inches='tight', dpi=300)
plt.savefig('../results/figures/AWIK_Ablation_Routing_Chart.png', bbox_inches='tight', dpi=300)
plt.show()
print("AWIK pipeline complete.")